# Step 1 — Prepare & enrich the contribution data

Loads the three input files from `source_data/`, enriches every contribution,
and writes the cleaned dataset that step 2 reads.

### What it does
1. Load `contributions.json`, `poster_submission_numbers.json`, and
   `manual_to_standard_institutions_dictionary.json`.
2. Flag each contribution's `type`: `"poster"` if its submission number is in
   the poster list, otherwise `"oral"` (a binary poster/talk split).
3. Attach each person's standardized `inst` (abbreviation) and `country` from
   the abbreviation map.
4. Write `step_1_output/contributions_rewritten.json` plus the deduplicated
   list of institutions seen in the data.

Run the cells in order.


In [ ]:
import json
from pathlib import Path
from pprint import pprint
from malapa_helpers import list_keys, iter_people

## Load the input files

Load the three JSON inputs and sanity-check their structure with `list_keys`.
The exploration cells below just print the shapes — they don't transform anything.


In [ ]:
# Set this to your JSON file path
JSON_FILE_PATH = Path("source_data/contributions.json").expanduser()
POSTER_PATH = Path("source_data/poster_submission_numbers.json").expanduser()
INST_PATH = Path("source_data/manual_to_standard_institutions_dictionary.json").expanduser()

In [ ]:
with JSON_FILE_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)
with POSTER_PATH.open("r", encoding="utf-8") as f:
    posters = json.load(f)
with INST_PATH.open("r", encoding="utf-8") as f:
    inst = json.load(f)

In [ ]:
print("Exploring contributions.json:")
list_keys(data)

Exploring contributions.json:


['[0].submission_number',
 '[0].id',
 '[0].url',
 '[0].title',
 '[0].speakers',
 '[0].speakers[0].name',
 '[0].speakers[0].manual_institution',
 '[0].authors',
 '[0].authors[0].name',
 '[0].authors[0].manual_institution',
 '[0].co_authors',
 '[0].description',
 '[0].presentation_materials']

In [ ]:
print("Exploring poster_submission_numbers.json:")
list_keys(posters)

Exploring poster_submission_numbers.json:


['[0].submission_number', '[0].type']

In [ ]:
print("Exploring manual_to_standard_institutions_dictionary.json:(note that this is a dictionary, not a list)")
print(f"Total institution keys loaded: {len(inst)}")
first_inst_key = next(iter(inst))
print(f"Example institution key: {first_inst_key}")
list_keys(inst[first_inst_key])

Exploring manual_to_standard_institutions_dictionary.json:(note that this is a dictionary, not a list)
Total institution keys loaded: 98
Example institution key: ASTeC


['inst', 'country']

# Merge missing info:
1. For each submission in `data`:
   * Add key `"type"` = `"poster"` if `"submission_number"` is listed in `posters`

2. For each person in `"speakers"`, `"authors"`, `"co_authors"`: 
   * Match **key** of `inst` to person's `"manual_institution"` in `data`
      * Add additional keys to each person in `data`: `"country"` and `"inst"`

> **Note:** institution matching is by exact name. A person whose
> `"manual_institution"` isn't a key in `manual_to_standard_institutions_dictionary.json` keeps no
> `inst`/`country` (a warning is printed) and is therefore excluded from the
> step-2 plots.


In [ ]:
poster_submission_numbers = set(p.get("submission_number") for p in posters if "submission_number" in p)
print(poster_submission_numbers)

for contribution in data:
    # set type based on poster list
    if contribution["submission_number"] in poster_submission_numbers:
        contribution["type"] = "poster"
    else:
        contribution["type"] = "oral"

    # add institution short name and country
    for person in iter_people(contribution):
        manual_institution = person["manual_institution"]
        try:
            this_persons_institution_info = inst[manual_institution]
        except KeyError:
            print(f"Institution {manual_institution} not found for person: {person['name']}")
            continue
        person["inst"] = this_persons_institution_info["inst"]
        person["country"] = this_persons_institution_info["country"]
    




{131, 133, 20, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 48, 51, 52, 53, 54, 57, 58, 65, 68, 70, 71, 72, 74, 76, 77, 78, 82, 84, 85, 88, 94, 97, 99, 100, 101, 102, 103, 105, 106, 108, 109, 110, 111, 112, 113, 115, 123, 125, 127}


# Save fixed file

In [ ]:
# Write the current `data` object back to JSON in the step 1 output folder.
output_dir = Path("step_1_output")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{JSON_FILE_PATH.stem}_rewritten.json"

with output_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Wrote JSON to: {output_path}")
print(f"Items written: {len(data) if isinstance(data, list) else 'n/a'}")

Wrote JSON to: step_1_output/contributions_rewritten.json
Items written: 94


## Save the list of institutions seen

The deduplicated `"ABBR, Country"` list of every institution that appears in the
contributions. This is the seed that was handed to Claude to build
`source_data/institution_locations.json`.


In [ ]:
institutions_seen = set()
for contribution in data:
    for person in iter_people(contribution):
        abbr = person.get("inst")
        country = person.get("country")
        if abbr and country:
            institutions_seen.add(f"{abbr}, {country}")

seen_path = output_dir / "institutions_seen.json"
with seen_path.open("w", encoding="utf-8") as f:
    json.dump(sorted(institutions_seen), f, indent=2, ensure_ascii=False)

print(f"Wrote {len(institutions_seen)} institutions to: {seen_path}")

Wrote 52 institutions to: step_1_output/institutions_seen.json


# Print some stats

In [ ]:
no_materials = []
multiple_materials = []
no_speakers = []
multiple_speakers = []
for contribution in data:
    if contribution["presentation_materials"] == []:
        no_materials.append(contribution)
    if len(contribution["presentation_materials"]) > 1:
        multiple_materials.append(contribution)
    if len(contribution["speakers"]) == 0:
        no_speakers.append(contribution)
    if len(contribution["speakers"]) > 1:
        multiple_speakers.append(contribution)

print(len(no_materials), "contributions with no presentation materials")
print(len(multiple_materials), "contributions with multiple presentation materials")
print(len(no_speakers), "contributions with no speakers")
print(len(multiple_speakers), "contributions with multiple speakers")

50 contributions with no presentation materials
3 contributions with multiple presentation materials
0 contributions with no speakers
7 contributions with multiple speakers


In [ ]:
print("Contributions with multiple presentation materials:")
for contribution in multiple_materials:
    print(f"\n\nsubmission_number: {contribution['submission_number']}, title: {contribution['title']}, materials:")
    pprint(contribution['presentation_materials'])

Contributions with multiple presentation materials:


submission_number: 18, title: Machine learning for accelerator design study and online tuning at NSLS-II, materials:
[{'filename': 'MaLAPA_Li.pptx',
  'url': 'https://indico.rcnp.osaka-u.ac.jp/event/2676/contributions/17581/attachments/10304/14636/MaLAPA_Li.pptx'},
 {'filename': 'MaLAPA_Li.pptx',
  'url': 'https://indico.rcnp.osaka-u.ac.jp/event/2676/contributions/17581/attachments/10304/14451/MaLAPA_Li.pptx'}]


submission_number: 43, title: The US DOE Multi-Office particle Accelerator Team (MOAT) project, materials:
[{'filename': 'MOAT_MALAPA_Vay.pdf',
  'url': 'https://indico.rcnp.osaka-u.ac.jp/event/2676/contributions/17595/attachments/10397/14609/MOAT_MALAPA_Vay.pdf'},
 {'filename': 'MOAT_MALAPA_Vay.pptx',
  'url': 'https://indico.rcnp.osaka-u.ac.jp/event/2676/contributions/17595/attachments/10397/14608/MOAT_MALAPA_Vay.pptx'}]


submission_number: 59, title: Predictive Modelling of Beam Charge: Improving Stability Through Data-

In [ ]:
for contribution in multiple_speakers:
    print(f"\n\nsubmission_number: {contribution['submission_number']}, title: {contribution['title']}, speakers:")
    for speaker in contribution['speakers']:
        print(f"  - {speaker['name']} ({speaker['inst']})")



submission_number: 14, title: femto-PIXAR: a self-supervised neural network method for reconstructing femtosecond X-ray free electron laser pulses, speakers:
  - Gesa Goetzke (DESY)
  - Rajan Plumley (SLAC)
  - Daniel Ratner (SLAC)


submission_number: 15, title: Virtual Diagnostics for the European XFEL, speakers:
  - Bianca Veglia (DESY)
  - Sergey Tomin (DESY)


submission_number: 76, title: Fast FEL Tuning using Feedback-Optimisation and Reinforcement Learning at the EuXFEL, speakers:
  - Christian Hespe (DESY)
  - Jan Kaiser (DESY)


submission_number: 77, title: Interpretable CNN Autoencoder Based Fault Diagnosis for the Optical Synchronization System of the European XFEL, speakers:
  - Jan Kaiser (DESY)
  - Christian Hespe (DESY)


submission_number: 78, title: Benchmarking and Optimizing Long-Term Shot-Synchronized Data Storage for Large-Scale Accelerator Diagnostics, speakers:
  - Jan Kaiser (DESY)
  - Christian Hespe (DESY)


submission_number: 97, title: Response Matrix Id